In [ ]:

# CELL 01: INITIALIZATION

from snowflake.snowpark import functions as F
from snowflake.snowpark.window import Window

# Database configuration
DATABASE = "RETAIL_MEDALLION_DB"

# Schema configuration
BRONZE = f"{DATABASE}.BRONZE"
SILVER = f"{DATABASE}.SILVER"
GOLD = f"{DATABASE}.GOLD"

# Select database
session.sql(f"USE DATABASE {DATABASE}").collect()

print("Environment initialized successfully")
print("Database:", DATABASE)


In [ ]:

# CELL 02: READ BRONZE DATA

customers = session.table(f"{BRONZE}.RAW_CUSTOMERS")
products = session.table(f"{BRONZE}.RAW_PRODUCTS")
orders = session.table(f"{BRONZE}.RAW_ORDERS")
order_items = session.table(f"{BRONZE}.RAW_ORDER_ITEMS")

print("Bronze data loaded")

print("Customers:", customers.count())
print("Products:", products.count())
print("Orders:", orders.count())
print("Order items:", order_items.count())

display(customers.limit(10))


In [ ]:

# CELL 03: DATA PROFILING

source_data = {
    "CUSTOMERS": customers,
    "PRODUCTS": products,
    "ORDERS": orders,
    "ORDER_ITEMS": order_items
}

for table_name, df in source_data.items():

    print(f"\n--- {table_name} ---")

    print("Total records:", df.count())

    for column in df.columns:
        null_count = df.filter(
            F.col(column).is_null()
        ).count()

        if null_count > 0:
            print(column, "NULL values:", null_count)


In [ ]:

# CELL 04: DUPLICATE CUSTOMER DETECTION

customer_window = (
    Window
    .partition_by("CUSTOMER_ID")
    .order_by(F.col("INGESTED_AT").desc())
)

customers_ranked = customers.with_column(
    "ROW_NUMBER",
    F.row_number().over(customer_window)
)

duplicate_customers = customers_ranked.filter(
    F.col("ROW_NUMBER") > 1
)

print("Duplicate customers:", duplicate_customers.count())

display(duplicate_customers)


In [ ]:

# CELL 05: CLEAN CUSTOMERS

clean_customers = (
    customers_ranked
    .filter(
        (F.col("ROW_NUMBER") == 1)
        & F.col("CUSTOMER_NAME").is_not_null()
        & F.col("EMAIL").is_not_null()
        & F.col("EMAIL").rlike(
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        )
    )
    .drop("ROW_NUMBER")
    .with_column(
        "CUSTOMER_NAME",
        F.trim(F.col("CUSTOMER_NAME"))
    )
    .with_column(
        "EMAIL",
        F.lower(F.trim(F.col("EMAIL")))
    )
    .with_column(
        "COUNTRY",
        F.initcap(F.trim(F.col("COUNTRY")))
    )
    .with_column(
        "CITY",
        F.initcap(F.trim(F.col("CITY")))
    )
)

print("Clean customers:", clean_customers.count())

display(clean_customers)


In [ ]:

# CELL 06: CLEAN PRODUCTS

clean_products = (
    products
    .filter(
        F.col("PRODUCT_NAME").is_not_null()
        & F.col("CATEGORY").is_not_null()
        & F.col("UNIT_PRICE").is_not_null()
        & (F.col("UNIT_PRICE") > 0)
    )
    .with_column(
        "PRODUCT_NAME",
        F.trim(F.col("PRODUCT_NAME"))
    )
    .with_column(
        "CATEGORY",
        F.upper(F.trim(F.col("CATEGORY")))
    )
)

print("Clean products:", clean_products.count())

display(clean_products)


In [ ]:

# CELL 07: CLEAN ORDERS

valid_customer_ids = clean_customers.select(
    "CUSTOMER_ID"
).distinct()

clean_orders = (
    orders
    .filter(
        F.col("ORDER_STATUS").is_not_null()
        & F.upper(F.col("ORDER_STATUS")).isin(
            ["COMPLETED", "PENDING", "CANCELLED"]
        )
    )
    .join(
        valid_customer_ids,
        "CUSTOMER_ID",
        "left_semi"
    )
    .with_column(
        "ORDER_STATUS",
        F.upper(F.trim(F.col("ORDER_STATUS")))
    )
)

print("Clean orders:", clean_orders.count())

display(clean_orders)


In [ ]:

# CELL 08: CLEAN ORDER ITEMS

valid_order_ids = clean_orders.select(
    "ORDER_ID"
).distinct()

valid_product_ids = clean_products.select(
    "PRODUCT_ID"
).distinct()

clean_order_items = (
    order_items
    .filter(
        F.col("QUANTITY").is_not_null()
        & (F.col("QUANTITY") > 0)
        & F.col("UNIT_PRICE").is_not_null()
        & (F.col("UNIT_PRICE") > 0)
    )
    .join(valid_order_ids, "ORDER_ID", "left_semi")
    .join(valid_product_ids, "PRODUCT_ID", "left_semi")
)

print("Clean order items:", clean_order_items.count())

display(clean_order_items)


In [ ]:

# CELL 09: SAVE SILVER TABLES
# Match source and target columns by column name

def save_table(df, table_name):

    # Clear existing target data
    session.sql(
        f"TRUNCATE TABLE {table_name}"
    ).collect()

    # Insert records, matching columns by name
    df.write.mode("append").save_as_table(
        table_name,
        column_order="name"
    )

    print("Successfully saved:", table_name)
    print("Rows inserted:", df.count())


save_table(clean_customers, f"{SILVER}.CLEAN_CUSTOMERS")
save_table(clean_products, f"{SILVER}.CLEAN_PRODUCTS")
save_table(clean_orders, f"{SILVER}.CLEAN_ORDERS")
save_table(clean_order_items, f"{SILVER}.CLEAN_ORDER_ITEMS")


In [ ]:

# CELL 10: BUILD GOLD DIMENSIONS

# Customer dimension
dim_customers = (
    clean_customers
    .select(
        "CUSTOMER_ID",
        "CUSTOMER_NAME",
        "EMAIL",
        "COUNTRY",
        "CITY"
    )
    .drop_duplicates(["CUSTOMER_ID"])
)

# Product dimension
dim_products = (
    clean_products
    .select(
        "PRODUCT_ID",
        "PRODUCT_NAME",
        "CATEGORY",
        "UNIT_PRICE"
    )
    .drop_duplicates(["PRODUCT_ID"])
)

# Save Gold dimensions
save_table(dim_customers, f"{GOLD}.DIM_CUSTOMERS")
save_table(dim_products, f"{GOLD}.DIM_PRODUCTS")

print("Gold dimensions created successfully.")


In [ ]:

# CELL 11: BUILD GOLD FACT SALES
# Avoid DataFrame aliases and ambiguous column names

# 1. Prepare order items
oi = clean_order_items.select(
    F.col("ORDER_ITEM_ID"),
    F.col("ORDER_ID").alias("ITEM_ORDER_ID"),
    F.col("PRODUCT_ID"),
    F.col("QUANTITY"),
    F.col("UNIT_PRICE")
)

# 2. Prepare orders
o = clean_orders.select(
    F.col("ORDER_ID").alias("HEADER_ORDER_ID"),
    F.col("CUSTOMER_ID"),
    F.col("ORDER_DATE"),
    F.col("ORDER_STATUS")
)

# 3. Prepare products
p = clean_products.select(
    F.col("PRODUCT_ID").alias("PRODUCT_LOOKUP_ID")
)

# 4. Join order items with orders
items_with_orders = oi.join(
    o,
    oi["ITEM_ORDER_ID"] == o["HEADER_ORDER_ID"],
    "inner"
)

# 5. Join with products
joined_sales = items_with_orders.join(
    p,
    items_with_orders["PRODUCT_ID"] == p["PRODUCT_LOOKUP_ID"],
    "inner"
)

# 6. Build final fact DataFrame
fact_sales = joined_sales.select(
    F.col("ORDER_ITEM_ID"),
    F.col("HEADER_ORDER_ID").alias("ORDER_ID"),
    F.col("CUSTOMER_ID"),
    F.col("PRODUCT_ID"),
    F.col("ORDER_DATE"),
    F.col("ORDER_STATUS"),
    F.col("QUANTITY"),
    F.col("UNIT_PRICE"),
    (
        F.col("QUANTITY") * F.col("UNIT_PRICE")
    ).alias("TOTAL_AMOUNT")
)

# 7. Save Gold fact table
save_table(
    fact_sales,
    f"{GOLD}.FACT_SALES"
)

print("Gold fact sales table created successfully.")


In [ ]:

# CELL 12: BUILD GOLD AGGREGATIONS

# ==========================================
# 1. SALES BY PRODUCT CATEGORY
# ==========================================

# Prepare product lookup
products_lookup = clean_products.select(
    F.col("PRODUCT_ID").alias("CATEGORY_PRODUCT_ID"),
    F.col("CATEGORY")
)

# Join fact sales with products
category_data = fact_sales.join(
    products_lookup,
    fact_sales["PRODUCT_ID"] == products_lookup["CATEGORY_PRODUCT_ID"],
    "inner"
)

# Aggregate sales by category
sales_by_category = (
    category_data
    .group_by("CATEGORY")
    .agg(
        F.count_distinct("ORDER_ID").alias("TOTAL_ORDERS"),
        F.sum("QUANTITY").alias("TOTAL_QUANTITY"),
        F.sum("TOTAL_AMOUNT").alias("TOTAL_REVENUE")
    )
)

# Save category aggregation
save_table(
    sales_by_category,
    f"{GOLD}.AGG_SALES_BY_CATEGORY"
)


# ==========================================
# 2. SALES BY CUSTOMER COUNTRY
# ==========================================

# Prepare customer lookup
customers_lookup = clean_customers.select(
    F.col("CUSTOMER_ID").alias("COUNTRY_CUSTOMER_ID"),
    F.col("COUNTRY")
)

# Join fact sales with customers
country_data = fact_sales.join(
    customers_lookup,
    fact_sales["CUSTOMER_ID"] == customers_lookup["COUNTRY_CUSTOMER_ID"],
    "inner"
)

# Aggregate sales by country
sales_by_country = (
    country_data
    .group_by("COUNTRY")
    .agg(
        F.count_distinct("ORDER_ID").alias("TOTAL_ORDERS"),
        F.sum("TOTAL_AMOUNT").alias("TOTAL_REVENUE")
    )
)

# Save country aggregation
save_table(
    sales_by_country,
    f"{GOLD}.AGG_SALES_BY_COUNTRY"
)

print("Gold aggregation tables created successfully.")


In [ ]:

# CELL 13: MEDALLION PIPELINE VALIDATION

# ==========================================
# 1. BRONZE, SILVER AND GOLD ROW COUNTS
# ==========================================

tables_to_check = [
    f"{BRONZE}.RAW_CUSTOMERS",
    f"{BRONZE}.RAW_PRODUCTS",
    f"{BRONZE}.RAW_ORDERS",
    f"{BRONZE}.RAW_ORDER_ITEMS",

    f"{SILVER}.CLEAN_CUSTOMERS",
    f"{SILVER}.CLEAN_PRODUCTS",
    f"{SILVER}.CLEAN_ORDERS",
    f"{SILVER}.CLEAN_ORDER_ITEMS",

    f"{GOLD}.DIM_CUSTOMERS",
    f"{GOLD}.DIM_PRODUCTS",
    f"{GOLD}.FACT_SALES",
    f"{GOLD}.AGG_SALES_BY_CATEGORY",
    f"{GOLD}.AGG_SALES_BY_COUNTRY"
]

print("MEDALLION PIPELINE - ROW COUNTS")
print("=" * 50)

for table_name in tables_to_check:
    count = session.table(table_name).count()
    print(f"{table_name}: {count}")


# ==========================================
# 2. GOLD FACT SALES
# ==========================================

print("\nGOLD FACT SALES")
print("=" * 50)

session.table(
    f"{GOLD}.FACT_SALES"
).show(10)


# ==========================================
# 3. SALES BY CATEGORY
# ==========================================

print("\nSALES BY CATEGORY")
print("=" * 50)

session.table(
    f"{GOLD}.AGG_SALES_BY_CATEGORY"
).sort(
    F.col("TOTAL_REVENUE").desc()
).show()


# ==========================================
# 4. SALES BY COUNTRY
# ==========================================

print("\nSALES BY COUNTRY")
print("=" * 50)

session.table(
    f"{GOLD}.AGG_SALES_BY_COUNTRY"
).sort(
    F.col("TOTAL_REVENUE").desc()
).show()


print("\nMEDALLION PIPELINE EXECUTION COMPLETED")
